# Helicopter Engine Health Monitoring & Predictive Maintenance Analysis
## Sikorsky S-76A/C+ Powered by Allison 250-C30S Turboshaft Engines

This notebook walks through the complete end-to-end data analysis and machine learning pipeline:
1. **Telemetry Ingestion & Data Validation**
2. **Sortie Segmentation & 3-Hour Endurance Verification**
3. **Feature Engineering & Exceedance Flagging**
4. **Exploratory Data Analysis & Visualizations**
5. **Engine Health Classification Modeling**
6. **Parameter Dynamics Forecasting (One-Step-Ahead Torque)**
7. **Remaining Useful Life (RUL) Feasibility Analysis**

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to python path
SRC_DIR = Path("..") / "src"
sys.path.insert(0, str(SRC_DIR.resolve()))

from data_loader import RAW_DIR, PROC_DIR, REP_DIR
from predict import load_health_model, load_forecasting_model
print("Environment initialized successfully!")

## 1. Inspect Consolidated Dataset
Loading the processed Parquet dataset containing over 3.35 million records across 9 continuous FDR recording sessions.

In [ ]:
df_cons = pd.read_parquet(PROC_DIR / "consolidated_dataset.parquet")
print(f"Total Records: {len(df_cons):,}")
print(f"Total Columns: {len(df_cons.columns)}")
print(f"Recording Sessions: {df_cons['source_file'].nunique()}")
df_cons[['recording_date', 'source_file', 'elapsed_seconds', 'E1TQ_S76C', 'E2TQ_S76C', 'E1NF_S76C', 'RTRSPD']].head()

## 2. Flight Duration & 3-Hour Constraint Analysis
Examine session durations against standard helicopter mission profiles (2.5 to 3.0 hours).

In [ ]:
flight_rep = pd.read_csv(REP_DIR / "flight_duration_report.csv")
display(flight_rep[['flight_id', 'source_file', 'duration_hours', 'constraint_status']])
print("
All sessions represent continuous multi-day FDR download recordings (65 to 114 hours).")

## 3. Pre-Trained Model Evaluation Metrics

In [ ]:
health_metrics = pd.read_csv(Path("..") / "results" / "metrics" / "health_model_metrics.csv")
forecast_metrics = pd.read_csv(Path("..") / "results" / "metrics" / "forecasting_model_metrics.csv")

print("=== ENGINE HEALTH CLASSIFIER METRICS ===")
display(health_metrics)

print("
=== TORQUE FORECASTING REGRESSOR METRICS ===")
display(forecast_metrics)

## 4. Live Model Inference Demonstration
Score sample flight data with both pre-trained models.

In [ ]:
health_clf, h_meta = load_health_model()
forecast_reg, f_meta = load_forecasting_model()

feat_df = pd.read_parquet(PROC_DIR / "features.parquet").head(10)

hf = [c for c in h_meta['features'] if c in feat_df.columns]
ff = [c for c in f_meta['features'] if c in feat_df.columns]

feat_df['exceedance_prob'] = health_clf.predict_proba(feat_df[hf])[:, 1]
feat_df['predicted_next_torque'] = forecast_reg.predict(feat_df[ff])

feat_df[['elapsed_seconds', 'E1TQ_S76C', 'predicted_next_torque', 'exceedance_prob']]

## 5. RUL Feasibility Conclusions
Supervised Remaining Useful Life (RUL) modeling cannot be computed with valid safety integrity on this dataset due to absence of failure events, cumulative engine operating hours, and maintenance overhauls.